<a href="https://colab.research.google.com/github/samreenfathima18/Guvi-Final-project/blob/main/02_Data_Merging_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pyarrow

**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import gc
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

**Mount Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Paths**

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Enterprise_Retail_Intelligence/data/"
OUTPUT_PATH = "/content/drive/MyDrive/Enterprise_Retail_Intelligence/outputs/"

**Read Data**

In [ ]:
sales = pd.read_csv(DATA_PATH + "sales_train_validation.csv")
calendar = pd.read_csv(DATA_PATH + "calendar.csv")
prices = pd.read_csv(DATA_PATH + "sell_prices.csv")

print("Datasets loaded successfully!")

print("Sales Shape    :", sales.shape)
print("Calendar Shape :", calendar.shape)
print("Prices Shape   :", prices.shape)

Datasets loaded successfully!
Sales Shape    : (30490, 1919)
Calendar Shape : (1969, 14)
Prices Shape   : (6841121, 4)


**Filter One Store**

In [ ]:
sales = sales[sales["store_id"] == "CA_1"].copy()

prices = prices[prices["store_id"] == "CA_1"].copy()

print(sales.shape)
print(prices.shape)

(3049, 1919)
(698412, 4)


**Convert Sales to Long Format**

In [ ]:
id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

sales_long = sales.melt(
    id_vars=id_cols,
    var_name="d",
    value_name="sales"
)

print(sales_long.shape)
sales_long.head()

(5832737, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


**Reduce Calendar Columns**

In [ ]:
calendar = calendar[
    [
        "d",
        "date",
        "wm_yr_wk",
        "weekday",
        "wday",
        "month",
        "year",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
        "snap_CA",
        "snap_TX",
        "snap_WI",
    ]
]

**Merge Calendar**

In [ ]:
sales_long = sales_long.merge(
    calendar,
    on="d",
    how="left"
)

print(sales_long.shape)

(5832737, 21)


In [ ]:
sales_long.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI'],
      dtype='object')

**Reduce Price Columns**

In [ ]:
prices = prices[
    [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "sell_price"
    ]
]

**Merge Prices**

In [ ]:
sales_long = sales_long.merge(
    prices,
    on=[
        "store_id",
        "item_id",
        "wm_yr_wk"
    ],
    how="left"
)

print(sales_long.shape)

(5832737, 22)


**Missing Values**

In [ ]:
sales_long.isnull().sum().sort_values(ascending=False).head(15)

,0
event_type_2,5820541
event_name_2,5820541
event_type_1,5363191
event_name_1,5363191
sell_price,1129842
id,0
state_id,0
store_id,0
cat_id,0
dept_id,0


**Fill Event Columns**

In [ ]:
event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

sales_long[event_cols] = sales_long[event_cols].fillna("None")

**Fill Price Columns**

In [ ]:
sales_long["sell_price"] = (
    sales_long
    .groupby(
        ["store_id", "item_id"]
    )["sell_price"]
    .ffill()
)

sales_long["sell_price"] = sales_long["sell_price"].fillna(0)

**Convert Date**

In [ ]:
sales_long["date"] = pd.to_datetime(
    sales_long["date"]
)

**Optimize Memory**

In [ ]:
for col in [
    "sales",
    "wday",
    "month",
    "year",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]:
    sales_long[col] = pd.to_numeric(
        sales_long[col],
        downcast="integer"
    )

sales_long["sell_price"] = pd.to_numeric(
    sales_long["sell_price"],
    downcast="float"
)

**Save Dataset**

In [ ]:
sales_long.to_parquet(
    OUTPUT_PATH + "processed_CA1.parquet",
    index=False
)

print("Saved Successfully!")

Saved Successfully!


In [ ]:
df = pd.read_parquet("/content/drive/MyDrive/Enterprise_Retail_Intelligence/outputs/processed_CA1.parquet")
print(df.shape)

(5832737, 22)
